1. Download the Zomato Restaurants Kaggle dataset, load it into a pandas DataFrame, and display the first 5 rows along with info() and describe() output.

In [10]:
import pandas as pd

df = pd.read_csv("zomato.csv")


print(df.info())

print(df.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rating                10 non-null     float64
 1   votes                 10 non-null     int64  
 2   online_order          10 non-null     int64  
 3   table_booking         10 non-null     int64  
 4   average_cost_for_two  10 non-null     int64  
dtypes: float64(1), int64(4)
memory usage: 532.0 bytes
None
          rating        votes  online_order  table_booking  \
count  10.000000    10.000000     10.000000      10.000000   
mean    4.200000  1570.000000      0.700000       0.400000   
std     0.365148  1098.281486      0.483046       0.516398   
min     3.700000   400.000000      0.000000       0.000000   
25%     3.925000   700.000000      0.250000       0.000000   
50%     4.150000  1150.000000      1.000000       0.000000   
75%     4.475000  2425.000000      1.000000     

In [5]:
print(df.head())


   rating  votes  online_order  table_booking  average_cost_for_two
0     4.5   2500             1              1                  1200
1     4.1    900             1              0                   700
2     3.9    650             0              0                   500
3     4.8   3500             1              1                  1800
4     4.2   1400             1              0                   900


2. Check the Zomato dataset for missing values and outliers in the 'cost' and 'rating' columns, and handle them appropriately using pandas and numpy.<br><br><em><strong>Hint:</strong> For outliers, try using the IQR method or visualize with a boxplot.</em>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rename columns
df = df.rename(columns={
    "rate":"rating",
    "approx_cost(for two people)":"cost",
    "cuisines":"cuisine"
})

# Remove '/5'
df["rating"] = df["rating"].astype(str).str.replace("/5","",regex=False)

df["rating"] = pd.to_numeric(df["rating"],errors="coerce")

# Remove commas
df["cost"] = df["cost"].astype(str).str.replace(",","")

df["cost"] = pd.to_numeric(df["cost"],errors="coerce")

print(df[["rating","cost"]].isnull().sum())

# Fill missing values
df["rating"].fillna(df["rating"].median(), inplace=True)
df["cost"].fillna(df["cost"].median(), inplace=True)

# Boxplots
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.boxplot(df["rating"])

plt.subplot(1,2,2)
plt.boxplot(df["cost"])

plt.show()

3. Encode the 'cuisine' and 'location' categorical columns using OneHotEncoder or LabelEncoder, and scale the 'cost' and 'rating' columns using StandardScaler.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

le = LabelEncoder()

df["cuisine"] = le.fit_transform(df["cuisine"].astype(str))

df["location"] = le.fit_transform(df["location"].astype(str))

scaler = StandardScaler()

df[["cost","rating"]] = scaler.fit_transform(
    df[["cost","rating"]]
)

4. Split the Zomato data into train and test sets (80/20), then train both a Logistic Regression and a Random Forest model to predict whether a restaurant has a rating above 4.0. Compare their ROC-AUC scores and print which model performed better.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score

# Target
df["target"] = (df["rating"] > 4).astype(int)

X = df[["cost","cuisine","location"]]
y = df["target"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

# Logistic Regression
lr = LogisticRegression()

lr.fit(X_train,y_train)

pred_lr = lr.predict_proba(X_test)[:,1]

auc_lr = roc_auc_score(y_test,pred_lr)

# Random Forest
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train,y_train)

pred_rf = rf.predict_proba(X_test)[:,1]

auc_rf = roc_auc_score(y_test,pred_rf)

print("Logistic Regression ROC-AUC :",auc_lr)

print("Random Forest ROC-AUC :",auc_rf)

if auc_rf > auc_lr:
    print("Random Forest performed better")
else:
    print("Logistic Regression performed better")

5. Use SMOTE to balance the classes if the number of restaurants with rating >4.0 is much lower than those with rating <=4.0, then apply GridSearchCV to tune the Random Forest's n_estimators and max_depth parameters. Print the best parameters and ROC-AUC score.<br><br><em><strong>Hint:</strong> Use imblearn's SMOTE and sklearn's GridSearchCV.</em>

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV

smote = SMOTE(random_state=42)

X_res,y_res = smote.fit_resample(X,y)

param_grid = {
    "n_estimators":[100,200],
    "max_depth":[10,20,None]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_res,y_res)

print("Best Parameters:")
print(grid.best_params_)

print("Best ROC-AUC:")
print(grid.best_score_)